In [ ]:
!pip install scikit-learn pandas numpy --quiet

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import json
import numpy as np
import pandas as pd
import pickle
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score
from sklearn.preprocessing import LabelEncoder
import re

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


In [ ]:
DATASET_PATH = "/content/drive/MyDrive/Inzira Project/full_dataset.json"

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)
print(f"✓ Dataset loaded: {len(df)} total pages")
print(f"  Opportunity     : {len(df[df['bert_label'] == 1])}")
print(f"  Not opportunity : {len(df[df['bert_label'] == 0])}")

✓ Dataset loaded: 551 total pages
  Opportunity     : 393
  Not opportunity : 158


In [ ]:
def extract_trust_features(row):
    text = str(row['text'])
    url  = str(row['url'])

    # Feature 1 — Domain authority signals
    trusted_domains = [
        'mastercardfdn.org', 'hec.gov.rw', 'rdb.rw', 'wda.gov.rw',
        'mifotra.gov.rw', 'alxafrica.com', 'coursera.org', 'edx.org',
        'opportunitiesforafricans.com', 'youthconnekt.net', '.gov',
        '.edu', '.org', 'unesco.org', 'undp.org', 'worldbank.org',
        'africaportal.org', 'scholarshipportal.com', 'jobs.rw'
    ]
    domain_score    = sum(1 for d in trusted_domains if d in url.lower())
    domain_score    = min(domain_score, 5)
    is_https        = 1 if url.startswith('https') else 0
    url_length      = min(len(url), 200)
    has_suspicious  = 1 if re.search(r'free-money|win-prize|click-here|guaranteed', url.lower()) else 0

    # Feature 2 — Content completeness signals
    has_deadline      = 1 if re.search(r'deadline|apply by|closing date|due date|applications close', text.lower()) else 0
    has_eligibility   = 1 if re.search(r'eligible|eligibility|requirements|qualifications|who can apply', text.lower()) else 0
    has_apply_link    = 1 if re.search(r'apply now|application form|register here|sign up|how to apply', text.lower()) else 0
    has_org_name      = 1 if re.search(r'foundation|institute|university|ministry|organization|centre|agency|council', text.lower()) else 0
    has_location      = 1 if re.search(r'rwanda|kigali|africa|remote|online|international', text.lower()) else 0
    has_amount        = 1 if re.search(r'\$|usd|rwf|stipend|funded|scholarship amount|grant', text.lower()) else 0
    has_duration      = 1 if re.search(r'months|weeks|years|duration|period|program length', text.lower()) else 0
    has_contact       = 1 if re.search(r'contact|email|phone|reach us|info@|support@', text.lower()) else 0
    content_score     = has_deadline + has_eligibility + has_apply_link + has_org_name + has_location + has_amount + has_duration + has_contact

    # Feature 3 — Linguistic quality signals
    words             = text.split()
    word_count        = len(words)
    avg_word_len      = np.mean([len(w) for w in words]) if words else 0
    has_min_content   = 1 if word_count >= 150 else 0
    has_numbers       = 1 if re.search(r'\d+', text) else 0
    has_dates         = 1 if re.search(r'\d{4}|\d{1,2}/\d{1,2}|january|february|march|april|may|june|july|august|september|october|november|december', text.lower()) else 0
    has_upper         = 1 if re.search(r'[A-Z]{2,}', text) else 0
    sentence_count    = len(re.findall(r'[.!?]', text))
    quality_score     = has_min_content + has_numbers + has_dates + has_upper + (1 if avg_word_len > 4 else 0)

    # Feature 4 — Spam and fraud signals
    spam_words        = ['congratulations', 'you have been selected', 'send money', 'western union', 'wire transfer', 'nigerian']
    spam_score        = sum(1 for w in spam_words if w in text.lower())
    exclamation_count = min(text.count('!'), 10)
    caps_ratio        = sum(1 for c in text if c.isupper()) / max(len(text), 1)

    return {
        'domain_score':      domain_score,
        'is_https':          is_https,
        'url_length':        url_length,
        'has_suspicious':    has_suspicious,
        'content_score':     content_score,
        'has_deadline':      has_deadline,
        'has_eligibility':   has_eligibility,
        'has_apply_link':    has_apply_link,
        'has_org_name':      has_org_name,
        'has_location':      has_location,
        'has_amount':        has_amount,
        'has_duration':      has_duration,
        'has_contact':       has_contact,
        'quality_score':     quality_score,
        'has_dates':         has_dates,
        'sentence_count':    min(sentence_count, 100),
        'word_count':        min(word_count, 1000),
        'avg_word_len':      round(avg_word_len, 2),
        'spam_score':        spam_score,
        'exclamation_count': exclamation_count,
        'caps_ratio':        round(caps_ratio, 4)
    }

print("Extracting trust features...")
features_list = [extract_trust_features(row) for _, row in df.iterrows()]
features_df   = pd.DataFrame(features_list)
labels        = df['bert_label'].tolist()

print(f"✓ Features extracted for {len(features_df)} pages")

Extracting trust features...
✓ Features extracted for 551 pages


In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    features_df, labels, test_size=0.30, random_state=42, stratify=labels
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"✓ Dataset split:")
print(f"  Training set   : {len(X_train)} pages")
print(f"  Validation set : {len(X_val)} pages")
print(f"  Test set       : {len(X_test)} pages")

✓ Dataset split:
  Training set   : 385 pages
  Validation set : 83 pages
  Test set       : 83 pages


In [ ]:
print("── TRAINING RANDOM FOREST ──────────────────────────────")

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight='balanced'
)
rf_model.fit(X_train, y_train)

print("✓ Random Forest trained")

# Validation check
val_preds   = rf_model.predict(X_val)
val_accuracy = accuracy_score(y_val, val_preds)
print(f"✓ Validation accuracy: {val_accuracy*100:.2f}%")

── TRAINING RANDOM FOREST ──────────────────────────────
✓ Random Forest trained
✓ Validation accuracy: 79.52%


In [ ]:
!pip install imbalanced-learn --quiet
print("✓ Done")

✓ Done


In [ ]:
from imblearn.over_sampling import SMOTE

# Balance the training set
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print(f"✓ Before balancing:")
print(f"  Opportunity     : {sum(y_train)}")
print(f"  Not Opportunity : {len(y_train) - sum(y_train)}")
print(f"\n✓ After balancing:")
print(f"  Opportunity     : {sum(y_train_balanced)}")
print(f"  Not Opportunity : {len(y_train_balanced) - sum(y_train_balanced)}")

# Retrain with balanced data
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_split=3,
    min_samples_leaf=1,
    random_state=42
)
rf_model.fit(X_train_balanced, y_train_balanced)
print("\n✓ Random Forest retrained on balanced data")

val_preds    = rf_model.predict(X_val)
val_accuracy = accuracy_score(y_val, val_preds)
print(f"✓ Validation accuracy: {val_accuracy*100:.2f}%")

✓ Before balancing:
  Opportunity     : 275
  Not Opportunity : 110

✓ After balancing:
  Opportunity     : 275
  Not Opportunity : 275

✓ Random Forest retrained on balanced data
✓ Validation accuracy: 77.11%


In [ ]:
print("── FINAL TEST SET RESULTS ──────────────────────────────")

test_preds    = rf_model.predict(X_test)
test_proba    = rf_model.predict_proba(X_test)[:, 1]
test_accuracy = accuracy_score(y_test, test_preds)
auc_roc       = roc_auc_score(y_test, test_proba)

print(f"\n  Test Accuracy  : {test_accuracy*100:.2f}%")
print(f"  AUC-ROC Score  : {auc_roc:.4f}")
print(f"\n  Target: AUC-ROC ≥ 0.85")

if auc_roc >= 0.85:
    print("  ✓ TARGET MET — Trust scorer is ready")
else:
    print("  ✗ Target not met")

print("\n── FULL CLASSIFICATION REPORT ──────────────────────────")
print(classification_report(
    y_test, test_preds,
    target_names=['Not Opportunity', 'Opportunity']
))

── FINAL TEST SET RESULTS ──────────────────────────────

  Test Accuracy  : 69.88%
  AUC-ROC Score  : 0.7895

  Target: AUC-ROC ≥ 0.85
  ✗ Target not met

── FULL CLASSIFICATION REPORT ──────────────────────────
                 precision    recall  f1-score   support

Not Opportunity       0.48      0.67      0.56        24
    Opportunity       0.84      0.71      0.77        59

       accuracy                           0.70        83
      macro avg       0.66      0.69      0.67        83
   weighted avg       0.74      0.70      0.71        83



In [ ]:
print("── FINAL TEST SET RESULTS ──────────────────────────────")

test_preds      = rf_model.predict(X_test)
test_proba      = rf_model.predict_proba(X_test)[:, 1]
test_accuracy   = accuracy_score(y_test, test_preds)
auc_roc         = roc_auc_score(y_test, test_proba)

print(f"\n  Test Accuracy  : {test_accuracy*100:.2f}%")
print(f"  AUC-ROC Score  : {auc_roc:.4f}")
print(f"\n  Target: AUC-ROC ≥ 0.85")

if auc_roc >= 0.85:
    print("  ✓ TARGET MET — Trust scorer is ready")
else:
    print("  ✗ Target not met")

print("\n── FULL CLASSIFICATION REPORT ──────────────────────────")
print(classification_report(
    y_test, test_preds,
    target_names=['Not Opportunity', 'Opportunity']
))

print("\n── FEATURE IMPORTANCE ──────────────────────────────────")
feature_names = features_df.columns.tolist()
importances   = rf_model.feature_importances_
for name, importance in sorted(zip(feature_names, importances), key=lambda x: -x[1]):
    print(f"  {name:20s} : {importance:.4f}")

── FINAL TEST SET RESULTS ──────────────────────────────

  Test Accuracy  : 69.88%
  AUC-ROC Score  : 0.7895

  Target: AUC-ROC ≥ 0.85
  ✗ Target not met

── FULL CLASSIFICATION REPORT ──────────────────────────
                 precision    recall  f1-score   support

Not Opportunity       0.48      0.67      0.56        24
    Opportunity       0.84      0.71      0.77        59

       accuracy                           0.70        83
      macro avg       0.66      0.69      0.67        83
   weighted avg       0.74      0.70      0.71        83


── FEATURE IMPORTANCE ──────────────────────────────────
  content_score        : 0.1333
  has_org_name         : 0.1109
  word_count           : 0.0970
  url_length           : 0.0944
  sentence_count       : 0.0886
  caps_ratio           : 0.0855
  has_deadline         : 0.0849
  avg_word_len         : 0.0844
  has_eligibility      : 0.0491
  has_apply_link       : 0.0327
  has_location         : 0.0272
  exclamation_count    : 0.0218


In [ ]:
os.makedirs("/content/drive/MyDrive/Inzira Project/models/trust_scorer", exist_ok=True)

with open("/content/drive/MyDrive/Inzira Project/models/trust_scorer/random_forest.pkl", "wb") as f:
    pickle.dump(rf_model, f)

print("✓ Random Forest trust scorer saved to Google Drive")
print("✓ Location: Inzira Project/models/trust_scorer/random_forest.pkl")

✓ Random Forest trust scorer saved to Google Drive
✓ Location: Inzira Project/models/trust_scorer/random_forest.pkl


In [ ]:
from transformers import BertTokenizer, BertModel
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

bert_model = BertModel.from_pretrained(
    "/content/drive/MyDrive/Inzira Project/models/bert_classifier"
)
bert_tokenizer = BertTokenizer.from_pretrained(
    "/content/drive/MyDrive/Inzira Project/models/bert_classifier"
)
bert_model = bert_model.to(device)
bert_model.eval()
print(f"✓ BERT model loaded on {device}")

Loading weights:   0%|          | 0/199 [00:02<?, ?it/s]

BertModel LOAD REPORT from: /content/drive/MyDrive/Inzira Project/models/bert_classifier
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ BERT model loaded on cpu


In [ ]:
def get_bert_embedding(text, model, tokenizer, device, max_length=256):
    encoding = tokenizer(
        text,
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    input_ids      = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        embedding = outputs.last_hidden_state[:, 0, :].cpu().numpy()
    return embedding[0]

print("Generating BERT embeddings for all pages...")
print("This will take about 5 to 10 minutes...")

df['text'] = df['text'].fillna('').astype(str)
df['text'] = df['text'].apply(lambda x: ' '.join(x.split()[:400]))

embeddings = []
for i, text in enumerate(df['text'].tolist()):
    emb = get_bert_embedding(text, bert_model, bert_tokenizer, device)
    embeddings.append(emb)
    if i % 50 == 0:
        print(f"  Processed {i}/{len(df)} pages")

embeddings_array = np.array(embeddings)
labels           = df['bert_label'].tolist()

print(f"✓ Embeddings generated: shape {embeddings_array.shape}")

# Save embeddings so you never need to regenerate
os.makedirs("/content/drive/MyDrive/Inzira Project/checkpoints", exist_ok=True)
np.save("/content/drive/MyDrive/Inzira Project/checkpoints/bert_embeddings.npy", embeddings_array)
np.save("/content/drive/MyDrive/Inzira Project/checkpoints/bert_labels.npy", np.array(labels))
print("✓ Embeddings saved to Google Drive")

Generating BERT embeddings for all pages...
This will take about 5 to 10 minutes...
  Processed 0/551 pages
  Processed 50/551 pages
  Processed 100/551 pages
  Processed 150/551 pages
  Processed 200/551 pages
  Processed 250/551 pages
  Processed 300/551 pages
  Processed 350/551 pages
  Processed 400/551 pages
  Processed 450/551 pages
  Processed 500/551 pages
  Processed 550/551 pages
✓ Embeddings generated: shape (551, 768)
✓ Embeddings saved to Google Drive


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

X_train, X_temp, y_train, y_temp = train_test_split(
    embeddings_array, labels, test_size=0.30, random_state=42, stratify=labels
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"✓ Split: Train {len(X_train)} | Val {len(X_val)} | Test {len(X_test)}")

# Balance with SMOTE
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)
print(f"✓ Balanced: {sum(y_train_balanced)} opportunity | {len(y_train_balanced) - sum(y_train_balanced)} not opportunity")

# Train Random Forest
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    random_state=42
)
rf_model.fit(X_train_balanced, y_train_balanced)
print("✓ Random Forest trained on BERT embeddings")

val_preds    = rf_model.predict(X_val)
val_accuracy = accuracy_score(y_val, val_preds)
print(f"✓ Validation accuracy: {val_accuracy*100:.2f}%")

✓ Split: Train 385 | Val 83 | Test 83
✓ Balanced: 275 opportunity | 275 not opportunity
✓ Random Forest trained on BERT embeddings
✓ Validation accuracy: 92.77%


In [ ]:
print("── FINAL TEST SET RESULTS ──────────────────────────────")

test_preds    = rf_model.predict(X_test)
test_proba    = rf_model.predict_proba(X_test)[:, 1]
test_accuracy = accuracy_score(y_test, test_preds)
auc_roc       = roc_auc_score(y_test, test_proba)

print(f"\n  Test Accuracy  : {test_accuracy*100:.2f}%")
print(f"  AUC-ROC Score  : {auc_roc:.4f}")
print(f"\n  Target: AUC-ROC ≥ 0.85")

if auc_roc >= 0.85:
    print("  ✓ TARGET MET — Trust scorer is ready")
else:
    print("  ✗ Target not met")

print("\n── FULL CLASSIFICATION REPORT ──────────────────────────")
print(classification_report(
    y_test, test_preds,
    target_names=['Not Opportunity', 'Opportunity']
))

── FINAL TEST SET RESULTS ──────────────────────────────

  Test Accuracy  : 83.13%
  AUC-ROC Score  : 0.9078

  Target: AUC-ROC ≥ 0.85
  ✓ TARGET MET — Trust scorer is ready

── FULL CLASSIFICATION REPORT ──────────────────────────
                 precision    recall  f1-score   support

Not Opportunity       0.71      0.71      0.71        24
    Opportunity       0.88      0.88      0.88        59

       accuracy                           0.83        83
      macro avg       0.79      0.79      0.79        83
   weighted avg       0.83      0.83      0.83        83



In [ ]:
import pickle

os.makedirs("/content/drive/MyDrive/Inzira Project/models/trust_scorer", exist_ok=True)

with open("/content/drive/MyDrive/Inzira Project/models/trust_scorer/random_forest.pkl", "wb") as f:
    pickle.dump(rf_model, f)

print("✓ Random Forest trust scorer saved to Google Drive")

✓ Random Forest trust scorer saved to Google Drive
